<a href="https://colab.research.google.com/github/karimmagdyabbasalsaadawy55555-cmd/NLP_Quiz-3/blob/main/NLP_Quiz_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:

import re
import nltk
import numpy as np
import pandas as pd

from collections import Counter, defaultdict
from nltk.corpus import movie_reviews, stopwords, wordnet
from nltk.tokenize import TreebankWordTokenizer
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk import pos_tag

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

nltk.download('movie_reviews', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

print("Libraries loaded successfully")


Libraries loaded successfully


In [13]:

fileids = movie_reviews.fileids()

texts = [movie_reviews.raw(fileid) for fileid in fileids]

labels = [
    movie_reviews.categories(fileid)[0]
    for fileid in fileids
]

data = pd.DataFrame({
    'text': texts,
    'label': labels
})

print(data.head())
print("\nDataset Shape:", data.shape)
print("\nLabel Distribution:")
print(data['label'].value_counts())


                                                text label
0  plot : two teen couples go to a church party ,...   neg
1  the happy bastard's quick movie review \ndamn ...   neg
2  it is movies like these that make a jaded movi...   neg
3   " quest for camelot " is warner bros . ' firs...   neg
4  synopsis : a mentally unstable man undergoing ...   neg

Dataset Shape: (2000, 2)

Label Distribution:
label
neg    1000
pos    1000
Name: count, dtype: int64


In [14]:

def clean_text(text):
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r"n't\b", " not", text, flags=re.IGNORECASE)
    text = re.sub(r"'m\b", " am", text, flags=re.IGNORECASE)
    text = re.sub(r"'re\b", " are", text, flags=re.IGNORECASE)
    text = re.sub(r"'ve\b", " have", text, flags=re.IGNORECASE)
    text = re.sub(r"'ll\b", " will", text, flags=re.IGNORECASE)
    text = re.sub(r"'d\b", " would", text, flags=re.IGNORECASE)
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

cleaned_texts = [clean_text(text) for text in texts]

print("Text cleaning completed")
print(cleaned_texts[0][:300])


Text cleaning completed
plot two teen couples go to a church party drink and then drive they get into an accident one of the guys dies but his girlfriend continues to see him in her life and has nightmares what s the deal watch the movie and sorta find out critique a mind fuck movie for the teen generation that touches on 


In [15]:

tokenizer = TreebankWordTokenizer()

sample_text = re.split(r'(?<=[.!?])\s+', texts[0].strip())[0]
cleaned_sample = clean_text(sample_text)

raw_tokens = tokenizer.tokenize(sample_text)
clean_tokens = tokenizer.tokenize(cleaned_sample)

print("Raw Tokens:")
print(raw_tokens)
print("Number of Raw Tokens:", len(raw_tokens))

print("\nCleaned Tokens:")
print(clean_tokens)
print("Number of Cleaned Tokens:", len(clean_tokens))


Raw Tokens:
['plot', ':', 'two', 'teen', 'couples', 'go', 'to', 'a', 'church', 'party', ',', 'drink', 'and', 'then', 'drive', '.']
Number of Raw Tokens: 16

Cleaned Tokens:
['plot', 'two', 'teen', 'couples', 'go', 'to', 'a', 'church', 'party', 'drink', 'and', 'then', 'drive']
Number of Cleaned Tokens: 13


In [16]:

stop_words = set(stopwords.words('english'))
stop_words.difference_update(['not', 'no', 'never'])

filtered_texts = []

for text in cleaned_texts:
    words = text.split()
    filtered = [word for word in words if word not in stop_words]
    filtered_texts.append(" ".join(filtered))

for i in range(5):
    sentence = re.split(r'(?<=[.!?])\s+', texts[i].strip())[0]
    before = clean_text(sentence)
    after = " ".join(
        word for word in before.split()
        if word not in stop_words
    )

    print("Before:", before)
    print("After:", after)
    print("-" * 50)


Before: plot two teen couples go to a church party drink and then drive
After: plot two teen couples go church party drink drive
--------------------------------------------------
Before: the happy bastard s quick movie review damn that y k bug
After: happy bastard quick movie review damn k bug
--------------------------------------------------
Before: it is movies like these that make a jaded movie viewer thankful for the invention of the timex indiglo watch
After: movies like make jaded movie viewer thankful invention timex indiglo watch
--------------------------------------------------
Before: quest for camelot is warner bros
After: quest camelot warner bros
--------------------------------------------------
Before: synopsis a mentally unstable man undergoing psychotherapy saves a boy from a potentially fatal accident and then falls in love with the boy s mother a fledgling restauranteur
After: synopsis mentally unstable man undergoing psychotherapy saves boy potentially fatal acci

In [17]:

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('R'):
        return wordnet.ADV
    return wordnet.NOUN

tagged_texts = [
    pos_tag(text.split())
    for text in filtered_texts
]

all_lemmas = []

for tagged_words in tagged_texts:
    for word, tag in tagged_words:
        lemma = lemmatizer.lemmatize(
            word,
            get_wordnet_pos(tag)
        )
        all_lemmas.append(lemma)

print("Most Common 10 Lemmas:")
print(Counter(all_lemmas).most_common(10))


Most Common 10 Lemmas:
[('not', 11811), ('film', 11150), ('movie', 6981), ('one', 6030), ('make', 4265), ('like', 3940), ('character', 3879), ('get', 3712), ('see', 3125), ('go', 3042)]


In [18]:

stemmer = PorterStemmer()

word_tag_counts = defaultdict(Counter)

for tagged_words in tagged_texts:
    for word, tag in tagged_words:
        word_tag_counts[word][tag] += 1

word_counts = Counter(
    word
    for text in filtered_texts
    for word in text.split()
)

candidate_words = [
    'movies', 'films', 'actors', 'actresses', 'characters',
    'stories', 'directors', 'performances', 'played',
    'playing', 'loved', 'hated', 'amazing', 'boring',
    'beautiful', 'interesting', 'effects', 'scenes',
    'better', 'worse', 'running', 'directed', 'makes',
    'seen', 'written'
]

selected_words = [
    word for word in candidate_words
    if word in word_counts
]

for word, count in word_counts.most_common():
    if len(selected_words) >= 15:
        break
    if word not in selected_words:
        selected_words.append(word)

selected_words = selected_words[:15]

results = []

for word in selected_words:
    tag = word_tag_counts[word].most_common(1)[0][0]
    lemma = lemmatizer.lemmatize(word, get_wordnet_pos(tag))
    stem = stemmer.stem(word)

    results.append([word, lemma, stem])

df = pd.DataFrame(
    results,
    columns=['Word', 'Lemma', 'Stem']
)

print(df.to_string(index=False))


        Word       Lemma     Stem
      movies       movie     movi
       films        film     film
      actors       actor    actor
   actresses     actress  actress
  characters   character  charact
     stories       story    stori
   directors    director director
performances performance  perform
      played        play     play
     playing        play     play
       loved        love     love
       hated         hat     hate
     amazing     amazing     amaz
      boring        bore     bore
   beautiful   beautiful   beauti


In [19]:

adjectives = []

for tagged_words in tagged_texts:
    for word, tag in tagged_words:
        if tag in ['JJ', 'JJR', 'JJS']:
            adjectives.append(word)

adjective_counts = Counter(adjectives)

print("Most Common 10 Adjectives:")
print(adjective_counts.most_common(10))


Most Common 10 Adjectives:
[('good', 2373), ('much', 1449), ('little', 1412), ('bad', 1396), ('new', 1292), ('many', 1269), ('great', 1149), ('big', 1065), ('best', 1048), ('first', 969)]


In [20]:

bigram_vectorizer = CountVectorizer(
    ngram_range=(2, 2)
)

bigram_matrix = bigram_vectorizer.fit_transform(cleaned_texts)

print("Matrix Shape:", bigram_matrix.shape)

bigram_counts = np.asarray(
    bigram_matrix.sum(axis=0)
).ravel()

bigram_names = bigram_vectorizer.get_feature_names_out()

top_bigrams = sorted(
    zip(bigram_names, bigram_counts),
    key=lambda x: x[1],
    reverse=True
)[:10]

print("\nTop 10 Bigrams:")

for bigram, count in top_bigrams:
    print(bigram, ":", count)


Matrix Shape: (2000, 492309)

Top 10 Bigrams:
of the : 8790
in the : 5832
the film : 4563
to the : 2786
to be : 2745
and the : 2539
on the : 2250
the movie : 2156
it is : 1945
with the : 1905


In [21]:

count_vectorizer = CountVectorizer()
tfidf_vectorizer = TfidfVectorizer()

count_matrix = count_vectorizer.fit_transform(cleaned_texts)
tfidf_matrix = tfidf_vectorizer.fit_transform(cleaned_texts)

count_values = np.asarray(count_matrix.sum(axis=0)).ravel()
tfidf_values = np.asarray(tfidf_matrix.sum(axis=0)).ravel()

count_names = count_vectorizer.get_feature_names_out()
tfidf_names = tfidf_vectorizer.get_feature_names_out()

count_top = sorted(
    zip(count_names, count_values),
    key=lambda x: x[1],
    reverse=True
)[:10]

tfidf_top = sorted(
    zip(tfidf_names, tfidf_values),
    key=lambda x: x[1],
    reverse=True
)[:10]

count_rank = {
    word: rank
    for rank, (word, value) in enumerate(count_top, 1)
}

tfidf_rank = {
    word: rank
    for rank, (word, value) in enumerate(tfidf_top, 1)
}

print("CountVectorizer Top 10:")
print(count_top)

print("\nTfidfVectorizer Top 10:")
print(tfidf_top)

print("\nWords that changed position:")

for word in count_rank:
    if word in tfidf_rank and count_rank[word] != tfidf_rank[word]:
        print(
            word,
            "Count Rank:", count_rank[word],
            "TF-IDF Rank:", tfidf_rank[word]
        )

print("\nOnly in CountVectorizer:")
print(set(count_rank) - set(tfidf_rank))

print("\nOnly in TF-IDF:")
print(set(tfidf_rank) - set(count_rank))


CountVectorizer Top 10:
[('the', np.int64(76574)), ('and', np.int64(35581)), ('of', np.int64(34126)), ('to', np.int64(31939)), ('is', np.int64(26071)), ('in', np.int64(21827)), ('it', np.int64(16110)), ('that', np.int64(15927)), ('not', np.int64(11811)), ('as', np.int64(11378))]

TfidfVectorizer Top 10:
[('the', np.float64(578.4531576877996)), ('and', np.float64(271.48895425344267)), ('of', np.float64(257.32658510789173)), ('to', np.float64(242.24435882570282)), ('is', np.float64(199.6532947510383)), ('in', np.float64(165.65439889146327)), ('it', np.float64(126.18493150413745)), ('that', np.float64(123.51193124168478)), ('not', np.float64(94.9273090829105)), ('as', np.float64(89.88216184167109))]

Words that changed position:

Only in CountVectorizer:
set()

Only in TF-IDF:
set()


In [22]:

X_train, X_test, y_train, y_test = train_test_split(
    cleaned_texts,
    labels,
    test_size=0.25,
    stratify=labels,
    random_state=42
)

model_vectorizer = CountVectorizer()

X_train_counts = model_vectorizer.fit_transform(X_train)
X_test_counts = model_vectorizer.transform(X_test)

model = MultinomialNB()

model.fit(X_train_counts, y_train)

predictions = model.predict(X_test_counts)

accuracy = accuracy_score(y_test, predictions)

print("Model Accuracy:", accuracy)

new_sentences = [
    "The movie was amazing and the acting was excellent.",
    "The film was boring and the story was terrible.",
    "The actors gave a wonderful performance."
]

new_cleaned = [
    clean_text(sentence)
    for sentence in new_sentences
]

new_counts = model_vectorizer.transform(new_cleaned)

new_predictions = model.predict(new_counts)

print("\nPredictions:")

for sentence, prediction in zip(new_sentences, new_predictions):
    print(sentence)
    print("Prediction:", prediction)
    print("-" * 40)


Model Accuracy: 0.824

Predictions:
The movie was amazing and the acting was excellent.
Prediction: pos
----------------------------------------
The film was boring and the story was terrible.
Prediction: neg
----------------------------------------
The actors gave a wonderful performance.
Prediction: pos
----------------------------------------
